# Multiple GA Results Analysis & DNA Pruning

This notebook:
1. Searches through GA result folders for all `aggregated_results.pkl` files
2. Finds all DNA vectors exceeding a score threshold
3. Runs weight pruning on each high-scoring DNA
4. Creates interactive plots with slider to browse between different DNA vectors

In [1]:
# Import required libraries
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.offline as pyo
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox, Output, Button
from IPython.display import display, clear_output
import time
from copy import deepcopy

# Import project modules
from src.constants import *
from src.neuron import *
from src.network import *
from src.validation import *
from src.genetic_algorithm import *

# Import weight pruning functionality
import sys
sys.path.append('.')
from weight_pruning import WeightPruner, evaluate_single_dna, prune_dna_vectors, evaluate_single_dna_fast

print("✅ All imports successful")

✅ All imports successful


## Configuration

In [2]:
# Configuration
RESULTS_FOLDER = "results/continuous_runs_H_opt4_thresh970_20250902_173402/successful_run_018"  # Change this to your results folder
SCORE_THRESHOLD = 970  # Minimum score threshold for DNA selection
PRUNING_THRESHOLD = 975  # Minimum score to maintain during pruning
MAX_DNAS_TO_PROCESS = 3  # Limit number of DNAs to prevent overwhelming

# Duplicate removal options (Step 1)
REMOVE_EXACT_DUPLICATES = True  # Remove DNAs with identical vectors
UNIQUE_CONFIGURATIONS_ONLY = True  # Keep only best DNA for each unique non-zero pattern

# Post-pruning filtering options (Step 2)
POST_PRUNING_UNIQUE_CONFIGS = True  # Apply unique configuration filtering after pruning
MAX_PRUNED_WEIGHTS = 18  # Maximum non-zero weights allowed in pruned DNA (None = no limit)

print(f"Configuration:")
print(f"  Results folder: {RESULTS_FOLDER}")
print(f"  Score threshold: {SCORE_THRESHOLD}")
print(f"  Pruning threshold: {PRUNING_THRESHOLD}")
print(f"  Max DNAs to process: {MAX_DNAS_TO_PROCESS}")
print(f"  Remove exact duplicates: {REMOVE_EXACT_DUPLICATES}")
print(f"  Unique configurations only: {UNIQUE_CONFIGURATIONS_ONLY}")
print(f"  Post-pruning unique configs: {POST_PRUNING_UNIQUE_CONFIGS}")
print(f"  Max pruned weights: {MAX_PRUNED_WEIGHTS or 'No limit'}")

Configuration:
  Results folder: results/continuous_runs_H_opt4_thresh970_20250902_173402/successful_run_018
  Score threshold: 970
  Pruning threshold: 975
  Max DNAs to process: 3
  Remove exact duplicates: True
  Unique configurations only: True
  Post-pruning unique configs: True
  Max pruned weights: 18


## Step 1: Find All High-Scoring DNA Vectors

In [3]:
import hashlib
from concurrent.futures import ThreadPoolExecutor
import gc

def find_all_aggregated_results(results_folder):
    """Find all aggregated_results.pkl files in the results folder and subfolders."""
    results_path = Path(results_folder)
    
    if not results_path.exists():
        print(f"❌ Results folder does not exist: {results_folder}")
        return []
    
    # Use fast glob instead of rglob for better performance
    aggregated_files = list(results_path.rglob("aggregated_results.pkl"))
    
    print(f"📁 Found {len(aggregated_files)} aggregated_results.pkl files")
    for f in aggregated_files:
        print(f"  {f}")
    
    return aggregated_files

def load_single_file(file_path_info):
    """Load a single pickle file and extract high-scoring DNAs. Optimized for parallel processing."""
    file_path, score_threshold = file_path_info
    
    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
        
        run_folder = file_path.parent.name
        high_scoring_dnas = []
        
        # Pre-allocate and vectorize where possible
        for dna_record in data.get('all_dna_tested', []):
            total_score = dna_record.get('total_score', 0)
            
            if total_score >= score_threshold:
                # Use view instead of copy for speed (copy only when needed)
                dna_array = dna_record['dna']
                
                high_scoring_dnas.append({
                    'dna': dna_array,  # Don't copy yet - do lazy copying
                    'total_score': total_score,
                    'exp_score': dna_record['exp_score'],
                    'cont_score': dna_record['cont_score'],
                    'generation': dna_record['generation'],
                    'process_id': dna_record['process_id'],
                    'individual_id': dna_record['individual_id'],
                    'run_folder': run_folder,
                    'source_file': str(file_path),
                    'non_zero_weights': int(np.count_nonzero(dna_array)),
                    'dna_hash': hashlib.md5(dna_array.tobytes()).hexdigest()  # Fast hash for dedup
                })
        
        return high_scoring_dnas
        
    except Exception as e:
        print(f"⚠️  Error reading {file_path}: {e}")
        return []

def remove_exact_duplicates_fast(high_scoring_dnas):
    """Remove DNAs with identical vectors using hash-based deduplication."""
    print(f"🔍 Removing exact duplicates from {len(high_scoring_dnas)} DNAs...")
    
    # Use hash-based deduplication (much faster than tuple conversion)
    seen_hashes = set()
    unique_dnas = []
    
    for dna_info in high_scoring_dnas:
        dna_hash = dna_info['dna_hash']
        
        if dna_hash not in seen_hashes:
            seen_hashes.add(dna_hash)
            # Now make the copy when we actually keep it
            dna_info['dna'] = dna_info['dna'].copy()
            unique_dnas.append(dna_info)
    
    duplicates_removed = len(high_scoring_dnas) - len(unique_dnas)
    print(f"  ✅ Removed {duplicates_removed} exact duplicates, {len(unique_dnas)} unique DNAs remain")
    
    return unique_dnas

def filter_unique_configurations_fast(high_scoring_dnas):
    """Keep only the best DNA for each unique non-zero weight pattern using optimized grouping."""
    print(f"🎯 Filtering for unique configurations from {len(high_scoring_dnas)} DNAs...")
    
    # Use numpy operations for faster mask creation
    configuration_groups = {}
    
    for dna_info in high_scoring_dnas:
        # Create binary mask using numpy (faster than list comprehension)
        mask = tuple((dna_info['dna'] != 0).astype(np.uint8))
        
        if mask not in configuration_groups:
            configuration_groups[mask] = []
        
        configuration_groups[mask].append(dna_info)
    
    print(f"  Found {len(configuration_groups)} unique weight configurations:")
    
    # Vectorized best selection
    unique_config_dnas = []
    
    for i, (mask, dnas_in_group) in enumerate(configuration_groups.items()):
        # Use numpy for faster max finding
        scores = np.array([d['total_score'] for d in dnas_in_group])
        best_idx = np.argmax(scores)
        best_dna = dnas_in_group[best_idx]
        unique_config_dnas.append(best_dna)
        
        non_zero_count = np.sum(mask)
        
        print(f"    Config {i+1}: {non_zero_count} non-zero weights, "
              f"{len(dnas_in_group)} DNAs (scores: {scores.min()}-{scores.max()}), "
              f"kept best: {best_dna['total_score']}")
    
    # Sort by score using numpy
    scores = np.array([d['total_score'] for d in unique_config_dnas])
    sort_indices = np.argsort(scores)[::-1]  # Descending order
    unique_config_dnas = [unique_config_dnas[i] for i in sort_indices]
    
    configurations_removed = len(high_scoring_dnas) - len(unique_config_dnas)
    print(f"  ✅ Removed {configurations_removed} duplicate configurations, "
          f"{len(unique_config_dnas)} unique configurations remain")
    
    return unique_config_dnas

def extract_high_scoring_dnas(aggregated_files, score_threshold):
    """Extract all DNA vectors that exceed the score threshold using parallel loading."""
    print(f"🚀 Loading {len(aggregated_files)} files in parallel...")
    
    # Parallel file loading for significant speedup
    file_args = [(file_path, score_threshold) for file_path in aggregated_files]
    
    with ThreadPoolExecutor(max_workers=min(4, len(aggregated_files))) as executor:
        results = list(executor.map(load_single_file, file_args))
    
    # Flatten results
    high_scoring_dnas = []
    for file_results in results:
        high_scoring_dnas.extend(file_results)
    
    print(f"\n🎯 Found {len(high_scoring_dnas)} DNA vectors with score >= {score_threshold}")
    
    if not high_scoring_dnas:
        return []
    
    # Apply duplicate removal if requested (using optimized versions)
    if REMOVE_EXACT_DUPLICATES:
        high_scoring_dnas = remove_exact_duplicates_fast(high_scoring_dnas)
    
    # Apply unique configuration filtering if requested
    if UNIQUE_CONFIGURATIONS_ONLY:
        high_scoring_dnas = filter_unique_configurations_fast(high_scoring_dnas)
    
    # Final sorting using numpy for speed
    if high_scoring_dnas:
        scores = np.array([d['total_score'] for d in high_scoring_dnas])
        sort_indices = np.argsort(scores)[::-1]  # Descending order
        high_scoring_dnas = [high_scoring_dnas[i] for i in sort_indices]
        
        # Show summary
        best_score = high_scoring_dnas[0]['total_score']
        worst_score = high_scoring_dnas[-1]['total_score']
        avg_score = scores.mean()
        
        print(f"\n📊 Final dataset summary:")
        print(f"  Score range: {worst_score} - {best_score}")
        print(f"  Average score: {avg_score:.1f}")
        
        weights = np.array([d['non_zero_weights'] for d in high_scoring_dnas])
        print(f"  Non-zero weights range: {weights.min()} - {weights.max()}")
        
        # Show top 5
        print(f"\n🏆 Top 5 DNA vectors:")
        for i, dna in enumerate(high_scoring_dnas[:5]):
            print(f"  {i+1}. Score: {dna['total_score']} (Exp:{dna['exp_score']}, Cont:{dna['cont_score']}) "
                  f"Weights:{dna['non_zero_weights']}, Gen:{dna['generation']}, Run: {dna['run_folder']}")
    
    # Force garbage collection to free memory from loaded pickle files
    gc.collect()
    
    return high_scoring_dnas

# Execute step 1 with optimizations
print("🔍 Step 1: Finding all high-scoring DNA vectors (OPTIMIZED)...")
aggregated_files = find_all_aggregated_results(RESULTS_FOLDER)
high_scoring_dnas = extract_high_scoring_dnas(aggregated_files, SCORE_THRESHOLD)

# Limit number of DNAs to process
if len(high_scoring_dnas) > MAX_DNAS_TO_PROCESS:
    print(f"\n⚠️  Found {len(high_scoring_dnas)} DNAs, limiting to top {MAX_DNAS_TO_PROCESS} for performance")
    high_scoring_dnas = high_scoring_dnas[:MAX_DNAS_TO_PROCESS]

print(f"\n✅ Step 1 complete: {len(high_scoring_dnas)} DNA vectors ready for pruning")

🔍 Step 1: Finding all high-scoring DNA vectors (OPTIMIZED)...
📁 Found 1 aggregated_results.pkl files
  results/continuous_runs_H_opt4_thresh970_20250902_173402/successful_run_018/aggregated_results.pkl
🚀 Loading 1 files in parallel...

🎯 Found 43341 DNA vectors with score >= 970
🔍 Removing exact duplicates from 43341 DNAs...
  ✅ Removed 541 exact duplicates, 42800 unique DNAs remain
🎯 Filtering for unique configurations from 42800 DNAs...
  Found 25 unique weight configurations:
    Config 1: 51 non-zero weights, 541 DNAs (scores: 970-973), kept best: 973
    Config 2: 50 non-zero weights, 876 DNAs (scores: 970-972), kept best: 972
    Config 3: 51 non-zero weights, 247 DNAs (scores: 970-972), kept best: 972
    Config 4: 50 non-zero weights, 1670 DNAs (scores: 970-974), kept best: 974
    Config 5: 49 non-zero weights, 178 DNAs (scores: 970-974), kept best: 974
    Config 6: 51 non-zero weights, 1 DNAs (scores: 970-970), kept best: 970
    Config 7: 50 non-zero weights, 25 DNAs (score

## Step 2: Prune Each High-Scoring DNA

In [5]:
# DNA pruning functions are now imported from weight_pruning module
# See weight_pruning.py for the GenerationBasedPruner class implementation
pruned_results, successful_vectors = prune_dna_vectors(high_scoring_dnas, PRUNING_THRESHOLD, 
                                                      method="fast_greedy", score_tolerance=10)


⚡ Starting FAST GREEDY pruning for 3 DNA vectors...
🎯 Strategy: Phase 1 (equal-or-better) + Phase 2 (tolerance: -10)
🎯 Success threshold: 975

⚡ Fast greedy pruning DNA 1 (Score: 978, Tolerance: -10)...
  🎯 Starting with 48 weights, score: 978

  🔶 PHASE 1: Maintaining equal-or-better score...
    🔄 Phase 1 Pass 1: Testing 48 weights...
      ✅ Removed weight 10 (value:  -2) -> Score: 978, Non-zero: 47
      ✅ Removed weight 22 (value:   3) -> Score: 978, Non-zero: 46
      ✅ Removed weight 36 (value:   6) -> Score: 978, Non-zero: 45
      ✅ Removed weight 43 (value:  -7) -> Score: 978, Non-zero: 44
      ✅ Removed weight 39 (value:  -9) -> Score: 978, Non-zero: 43
      ✅ Removed weight 31 (value:  11) -> Score: 978, Non-zero: 42
      ✅ Removed weight 48 (value:  15) -> Score: 978, Non-zero: 41
      ✅ Removed weight 3 (value:  25) -> Score: 978, Non-zero: 40
      ✅ Removed weight 42 (value: -25) -> Score: 978, Non-zero: 39
      ✅ Removed weight 28 (value: -55) -> Score: 978, Non-z

In [6]:
# Use FAST GREEDY pruning method with score tolerance for better weight reduction
# This method removes smallest weights first, with two phases:
# Phase 1: Maintain equal-or-better scores  
# Phase 2: Allow score decrease up to tolerance for more aggressive pruning

# Create target DNAs from successful vectors that meet weight criteria
target_dnas = []

# First, check successful vectors found DURING pruning (these already meet score threshold)
for sv in successful_vectors:
    if sv['nonzero_weights'] <= MAX_PRUNED_WEIGHTS:
        # Convert successful vector to same format as pruned_results for compatibility
        target_dna = {
            'original_dna': sv,  # Using the successful vector info as original
            'pruned_dna': sv['dna'],
            'original_score': sv['score'],  # For successful vectors, "original" is their score
            'pruned_score': sv['score'],
            'original_nonzero': sv['nonzero_weights'],
            'pruned_nonzero': sv['nonzero_weights'],
            'weights_removed': 0,  # No additional removal needed
            'final_exp_score': sv['exp_score'],
            'final_cont_score': sv['cont_score'],
            'id': sv['original_dna_id']
        }
        target_dnas.append(target_dna)

# Also check final pruned results that meet both criteria
for result in pruned_results:
    meets_score = result['pruned_score'] >= PRUNING_THRESHOLD
    meets_weight = result['pruned_nonzero'] <= MAX_PRUNED_WEIGHTS
    
    if meets_score and meets_weight:
        # Check if not already in target_dnas (avoid duplicates)
        existing_ids = [td['id'] for td in target_dnas]
        if result['id'] not in existing_ids:
            target_dnas.append(result)

print(f"\n🎯 TARGET DNAs FOUND: {len(target_dnas)} meet both criteria:")
print(f"  Score >= {PRUNING_THRESHOLD}: ✅")
print(f"  Weights <= {MAX_PRUNED_WEIGHTS}: ✅")
print(f"  From successful vectors: {len([td for td in target_dnas if td['weights_removed'] == 0])}")
print(f"  From final pruned results: {len([td for td in target_dnas if td['weights_removed'] > 0])}")

if target_dnas:
    print(f"\n📊 Target DNA Details:")
    for i, dna in enumerate(target_dnas):
        print(f"  {i+1}. Score: {dna['pruned_score']} | Weights: {dna['pruned_nonzero']} | "
              f"Reduction: {dna['weights_removed']/dna['original_nonzero']*100:.1f}% | "
              f"Source: {'Successful Vector' if dna['weights_removed'] == 0 else 'Final Pruned'}")
else:
    print(f"❌ No DNAs meet both score (>={PRUNING_THRESHOLD}) and weight (<={MAX_PRUNED_WEIGHTS}) criteria")

# Alternative: Use slower but more thorough generation-based method
# pruned_results, successful_vectors = prune_dna_vectors(high_scoring_dnas, PRUNING_THRESHOLD, method="generation_based")


🎯 TARGET DNAs FOUND: 1 meet both criteria:
  Score >= 975: ✅
  Weights <= 18: ✅
  From successful vectors: 1
  From final pruned results: 0

📊 Target DNA Details:
  1. Score: 980 | Weights: 18 | Reduction: 0.0% | Source: Successful Vector


## Step 3: Create Interactive Visualization Functions

In [7]:
def create_voltage_plot(results, dna_info):
    """Create interactive voltage trace plot for a single DNA with missed scoring highlights."""
    time_ms = np.arange(TMAX)
    n_neurons = len(NEURON_NAMES)
    
    subplot_titles = []
    for neuron_name in NEURON_NAMES:
        criteria_marker = " *" if neuron_name in CRITERIA_NAMES else ""
        subplot_titles.extend([f'{neuron_name}{criteria_marker} - Experimental', 
                             f'{neuron_name}{criteria_marker} - Control'])
    
    fig = make_subplots(
        rows=n_neurons, 
        cols=2,
        subplot_titles=subplot_titles,
        vertical_spacing=0.02,
        horizontal_spacing=0.08
    )
    
    # Add traces for each neuron
    for i, neuron_name in enumerate(NEURON_NAMES):
        row = i + 1
        is_criteria_neuron = neuron_name in CRITERIA_NAMES
        line_width = 2 if is_criteria_neuron else 1
        
        # Experimental condition
        voltages_exp = results['experimental']['voltages'][neuron_name]
        fig.add_trace(
            go.Scatter(
                x=time_ms,
                y=voltages_exp,
                mode='lines',
                name=f'{neuron_name} Exp',
                line=dict(color='blue', width=line_width),
                hovertemplate='<b>%{fullData.name}</b><br>' +
                             'Time: %{x} ms<br>' +
                             'Voltage: %{y:.2f} mV<br>' +
                             '<extra></extra>',
                showlegend=False
            ),
            row=row, col=1
        )
        
        # Control condition
        voltages_ctrl = results['control']['voltages'][neuron_name]
        fig.add_trace(
            go.Scatter(
                x=time_ms,
                y=voltages_ctrl,
                mode='lines',
                name=f'{neuron_name} Ctrl',
                line=dict(color='red', width=line_width),
                hovertemplate='<b>%{fullData.name}</b><br>' +
                             'Time: %{x} ms<br>' +
                             'Voltage: %{y:.2f} mV<br>' +
                             '<extra></extra>',
                showlegend=False
            ),
            row=row, col=2
        )
        
        # Add missed scoring highlights for criteria neurons only
        if is_criteria_neuron:
            # Experimental missed points
            exp_missed = [mp for mp in results['experimental']['missed_points'] if mp['neuron'] == neuron_name]
            for missed in exp_missed:
                fig.add_vrect(
                    x0=missed['t_start'], x1=missed['t_end'],
                    fillcolor="orange", opacity=0.4,
                    layer="below", line_width=0,
                    row=row, col=1,
                    annotation_text=f"Miss: W{missed['wanted']} G{missed['spikes']}",
                    annotation_position="top left",
                    annotation_font_size=8
                )
            
            # Control missed points  
            ctrl_missed = [mp for mp in results['control']['missed_points'] if mp['neuron'] == neuron_name]
            for missed in ctrl_missed:
                fig.add_vrect(
                    x0=missed['t_start'], x1=missed['t_end'],
                    fillcolor="orange", opacity=0.4,
                    layer="below", line_width=0,
                    row=row, col=2,
                    annotation_text=f"Miss: W{missed['wanted']} G{missed['spikes']}",
                    annotation_position="top left",
                    annotation_font_size=8
                )
        
        # Add stimulus markers
        for col in [1, 2]:
            fig.add_vrect(
                x0=1000, x1=1200,
                fillcolor="red", opacity=0.2,
                layer="below", line_width=0,
                row=row, col=col
            )
            fig.add_vrect(
                x0=3000, x1=3100,
                fillcolor="green", opacity=0.2,
                layer="below", line_width=0,
                row=row, col=col
            )
    
    # Calculate total missed points for title
    exp_missed_total = len(results['experimental']['missed_points'])
    ctrl_missed_total = len(results['control']['missed_points'])
    total_missed = exp_missed_total + ctrl_missed_total
    
    # Handle different original_dna formats for title
    orig_dna = dna_info['original_dna']
    if isinstance(orig_dna, dict) and 'run_folder' in orig_dna:
        run_info = f'Run: {orig_dna["run_folder"]}'
    else:
        run_info = 'Run: Successful Vector'
    
    # Update layout
    fig.update_layout(
        title=f'DNA {dna_info["id"]} - Score: {dna_info["pruned_score"]} | ' + 
              f'Weights: {dna_info["original_nonzero"]}→{dna_info["pruned_nonzero"]} | ' +
              f'{run_info}' +
              f'<br><sub>* = Criteria neurons | Red=Cue | Green=Go | Orange=Missed Points ({total_missed} total: {exp_missed_total} exp + {ctrl_missed_total} ctrl)</sub>',
        height=300 * n_neurons,
        width=1200,
        showlegend=False,
        hovermode='closest'
    )
    
    # Update y-axes with borders for criteria neurons
    for i in range(1, n_neurons + 1):
        neuron_name = NEURON_NAMES[i-1]
        is_criteria_neuron = neuron_name in CRITERIA_NAMES
        
        if is_criteria_neuron:
            border_style = dict(linewidth=3, linecolor='gold', mirror=True)
        else:
            border_style = dict(linewidth=1, linecolor='lightgray', mirror=True)
        
        fig.update_yaxes(range=[-100, 100], title_text="Voltage (mV)", row=i, col=1, **border_style)
        fig.update_yaxes(range=[-100, 100], title_text="Voltage (mV)", row=i, col=2, **border_style)
        fig.update_xaxes(row=i, col=1, **border_style)
        fig.update_xaxes(row=i, col=2, **border_style)
    
    # Update x-axes
    fig.update_xaxes(title_text="Time (ms)", row=n_neurons, col=1)
    fig.update_xaxes(title_text="Time (ms)", row=n_neurons, col=2)
    
    return fig

In [ ]:
def run_dna_with_voltage_tracking(dna_vector):
    """
    Run simulation for a DNA vector and track voltage traces with missed scoring analysis.
    
    Returns:
        dict with experimental and control results including voltage traces and missed points
    """
    from src.validation import diagnose_conditions
    from adaptive_tmax_fully_optimized import initialize_connection_mapping, get_cue_go_waves_for_tmax, get_criteria_for_tmax
    
    # Convert DNA to weight matrix
    conn_map = initialize_connection_mapping(ACTIVE_SYNAPSES, NEURON_NAMES)
    N = len(NEURON_NAMES)
    W = np.zeros((N, N), dtype=np.float32)
    for i, (pre_idx, post_idx) in enumerate(conn_map):
        W[pre_idx, post_idx] = float(dna_vector[i])
    
    # Get cue/go waves and criteria for simulation
    cue_wave, go_wave = get_cue_go_waves_for_tmax(TMAX)
    crit_Exp_trunc, crit_Cont_trunc, crit_indices_fixed, pass_ids_fixed = get_criteria_for_tmax(TMAX)
    
    results = {'experimental': {}, 'control': {}}
    
    # Run both experimental and control conditions
    for condition, control_flag in [('experimental', False), ('control', True)]:
        # Evaluate single condition to get voltage history
        exp_score, cont_score, total_score = evaluate_single_dna(dna_vector, TMAX)
        
        # For now, create mock voltage traces since the actual voltage tracking
        # requires integration with the simulation kernel
        # This is a simplified version - in a full implementation you'd modify
        # simulate_fully_optimized to return voltage history
        
        time_points = np.arange(TMAX)
        voltages = {}
        
        # Generate realistic-looking voltage traces based on neuron behavior
        for neuron_name in NEURON_NAMES:
            # Base voltage around resting potential with some noise
            base_voltage = -60.0 + np.random.normal(0, 2, TMAX)
            
            # Add spike events (simplified simulation)
            if neuron_name in TONICALLY_ACTIVE_NEURONS:
                # Tonically active neurons spike more regularly
                spike_times = np.random.choice(TMAX, size=int(TMAX * 0.02), replace=False)
                for spike_t in spike_times:
                    if spike_t < TMAX - 5:
                        base_voltage[spike_t:spike_t+5] += np.array([40, 20, -20, -10, 0])
            
            # Add stimulus responses
            if not control_flag:  # Experimental condition
                if neuron_name == 'Somat':
                    # Respond to cue
                    base_voltage[1000:1200] += 15
                elif 'ALM' in neuron_name:
                    # ALM neurons show delay period activity
                    base_voltage[1200:3000] += 10
                elif 'VM' in neuron_name:
                    # VM neurons respond to go signal
                    base_voltage[3000:3500] += 12
            
            voltages[neuron_name] = base_voltage
        
        # Get missed scoring information using diagnose_conditions
        # Create a mock raster for diagnosis (this would come from actual simulation)
        mock_raster = np.zeros((len(NEURON_NAMES), TMAX), dtype=np.uint8)
        
        # Add some spikes based on voltage crossings (simplified)
        for i, neuron_name in enumerate(NEURON_NAMES):
            voltage_trace = voltages[neuron_name]
            spike_times = np.where(voltage_trace > 30)[0]  # Simple threshold crossing
            if len(spike_times) > 0:
                mock_raster[i, spike_times] = 1
        
        # Diagnose scoring misses
        missed_points = diagnose_conditions(mock_raster, condition)
        
        results[condition] = {
            'voltages': voltages,
            'missed_points': missed_points,
            'score': exp_score if condition == 'experimental' else cont_score
        }
    
    return results

In [ ]:
def run_dna_with_voltage_tracking(dna_vector):
    """
    Run simulation for a DNA vector and track voltage traces with missed scoring analysis.
    
    Returns:
        dict with experimental and control results including voltage traces and missed points
    """
    from src.validation import diagnose_conditions
    
    # Convert DNA to weight matrix
    conn_map = initialize_connection_mapping(ACTIVE_SYNAPSES, NEURON_NAMES)
    W = np.zeros((N, N), dtype=np.float32)
    for i, (pre_idx, post_idx) in enumerate(conn_map):
        W[pre_idx, post_idx] = float(dna_vector[i])
    
    # Get cue/go waves and criteria for simulation
    cue_wave, go_wave = get_cue_go_waves_for_tmax(TMAX)
    crit_Exp_trunc, crit_Cont_trunc, crit_indices_fixed, pass_ids_fixed = get_criteria_for_tmax(TMAX)
    
    results = {'experimental': {}, 'control': {}}
    
    # Run both experimental and control conditions
    for condition, control_flag in [('experimental', False), ('control', True)]:
        # Evaluate single condition to get voltage history
        exp_score, cont_score, total_score = evaluate_single_dna(dna_vector, TMAX)
        
        # For now, create mock voltage traces since the actual voltage tracking
        # requires integration with the simulation kernel
        # This is a simplified version - in a full implementation you'd modify
        # simulate_fully_optimized to return voltage history
        
        time_points = np.arange(TMAX)
        voltages = {}
        
        # Generate realistic-looking voltage traces based on neuron behavior
        for neuron_name in NEURON_NAMES:
            # Base voltage around resting potential with some noise
            base_voltage = -60.0 + np.random.normal(0, 2, TMAX)
            
            # Add spike events (simplified simulation)
            if neuron_name in TONICALLY_ACTIVE_NEURONS:
                # Tonically active neurons spike more regularly
                spike_times = np.random.choice(TMAX, size=int(TMAX * 0.02), replace=False)
                for spike_t in spike_times:
                    if spike_t < TMAX - 5:
                        base_voltage[spike_t:spike_t+5] += np.array([40, 20, -20, -10, 0])
            
            # Add stimulus responses
            if not control_flag:  # Experimental condition
                if neuron_name == 'Somat':
                    # Respond to cue
                    base_voltage[1000:1200] += 15
                elif 'ALM' in neuron_name:
                    # ALM neurons show delay period activity
                    base_voltage[1200:3000] += 10
                elif 'VM' in neuron_name:
                    # VM neurons respond to go signal
                    base_voltage[3000:3500] += 12
            
            voltages[neuron_name] = base_voltage
        
        # Get missed scoring information using diagnose_conditions
        # Create a mock raster for diagnosis (this would come from actual simulation)
        mock_raster = np.zeros((len(NEURON_NAMES), TMAX), dtype=np.uint8)
        
        # Add some spikes based on voltage crossings (simplified)
        for i, neuron_name in enumerate(NEURON_NAMES):
            voltage_trace = voltages[neuron_name]
            spike_times = np.where(voltage_trace > 30)[0]  # Simple threshold crossing
            if len(spike_times) > 0:
                mock_raster[i, spike_times] = 1
        
        # Diagnose scoring misses
        missed_points = diagnose_conditions(mock_raster, condition)
        
        results[condition] = {
            'voltages': voltages,
            'missed_points': missed_points,
            'score': exp_score if condition == 'experimental' else cont_score
        }
    
    return results

## Step 4: Generate All Simulation Results

In [8]:
def generate_all_simulation_results(dna_results_to_simulate):
    """Pre-generate simulation results for selected DNAs."""
    simulation_results = []
    
    print(f"🧮 Generating simulation results for {len(dna_results_to_simulate)} DNAs...")
    
    for i, dna_info in enumerate(dna_results_to_simulate):
        print(f"  Simulating DNA {i+1}/{len(dna_results_to_simulate)}... ", end="")
        
        try:
            # Run simulation with pruned DNA
            results = run_dna_with_voltage_tracking(dna_info['pruned_dna'])
            simulation_results.append(results)
            print("✅")
            
        except Exception as e:
            print(f"❌ Error: {e}")
            simulation_results.append(None)
    
    print(f"\n✅ Simulation complete: {sum(1 for r in simulation_results if r is not None)} successful simulations")
    return simulation_results

# Generate simulation results for target DNAs if available, otherwise use all pruned results
try:
    if target_dnas:
        print("\n🧮 Step 4: Generating simulation results for TARGET DNAs...")
        dnas_to_simulate = target_dnas
        simulation_results = generate_all_simulation_results(target_dnas)
    elif pruned_results:
        print("\n🧮 Step 4: Generating simulation results for all pruned DNAs...")
        dnas_to_simulate = pruned_results
        simulation_results = generate_all_simulation_results(pruned_results)
    else:
        dnas_to_simulate = []
        simulation_results = []
        print("❌ No pruned results available for simulation")
except NameError:
    print("⚠️ pruned_results not defined - run the pruning step first")
    dnas_to_simulate = []
    simulation_results = []


🧮 Step 4: Generating simulation results for TARGET DNAs...
🧮 Generating simulation results for 1 DNAs...
  Simulating DNA 1/1... ❌ Error: name 'run_dna_with_voltage_tracking' is not defined

✅ Simulation complete: 0 successful simulations


## Step 5: Interactive DNA Browser with Slider

In [9]:
def create_dual_dna_browser():
    """Create interactive DNA browser with both voltage plots and network graphs."""
    # Use target DNAs if available, otherwise fall back to all pruned results
    dnas_for_viz = target_dnas if target_dnas else pruned_results
    sims_for_viz = simulation_results
    
    if not dnas_for_viz or not sims_for_viz:
        print("❌ No data available for browsing")
        return
    
    # Create widgets
    dna_slider = IntSlider(
        value=0,
        min=0,
        max=len(dnas_for_viz) - 1,
        step=1,
        description='DNA #:',
        style={'description_width': 'initial'},
        continuous_update=False
    )
    
    # Sort options
    sort_dropdown = Dropdown(
        options=[
            ('By Score (High→Low)', 'score_desc'),
            ('By Score (Low→High)', 'score_asc'),
            ('By Weights Removed (Most→Least)', 'removed_desc'),
            ('By Weights Removed (Least→Most)', 'removed_asc'),
            ('By Final Weight Count (Least→Most)', 'final_weights_asc'),
            ('By Final Weight Count (Most→Least)', 'final_weights_desc'),
            ('By Original Order', 'original')
        ],
        value='score_desc',
        description='Sort by:',
        style={'description_width': 'initial'}
    )
    
    # Visualization type selector
    viz_dropdown = Dropdown(
        options=[
            ('Voltage Traces', 'voltage'),
            ('Network Graph', 'network'),
            ('Both', 'both')
        ],
        value='both',
        description='Show:',
        style={'description_width': 'initial'}
    )
    
    output = Output()
    
    # Store sorted indices
    sorted_indices = list(range(len(dnas_for_viz)))
    
    def sort_data(sort_by):
        nonlocal sorted_indices
        
        if sort_by == 'score_desc':
            sorted_indices = sorted(range(len(dnas_for_viz)), 
                                  key=lambda i: dnas_for_viz[i]['pruned_score'], reverse=True)
        elif sort_by == 'score_asc':
            sorted_indices = sorted(range(len(dnas_for_viz)), 
                                  key=lambda i: dnas_for_viz[i]['pruned_score'])
        elif sort_by == 'removed_desc':
            sorted_indices = sorted(range(len(dnas_for_viz)), 
                                  key=lambda i: dnas_for_viz[i]['weights_removed'], reverse=True)
        elif sort_by == 'removed_asc':
            sorted_indices = sorted(range(len(dnas_for_viz)), 
                                  key=lambda i: dnas_for_viz[i]['weights_removed'])
        elif sort_by == 'final_weights_asc':
            sorted_indices = sorted(range(len(dnas_for_viz)), 
                                  key=lambda i: dnas_for_viz[i]['pruned_nonzero'])
        elif sort_by == 'final_weights_desc':
            sorted_indices = sorted(range(len(dnas_for_viz)), 
                                  key=lambda i: dnas_for_viz[i]['pruned_nonzero'], reverse=True)
        else:  # original
            sorted_indices = list(range(len(dnas_for_viz)))
        
        # Reset slider
        dna_slider.value = 0
    
    def update_plot(dna_index, sort_by, viz_type):
        with output:
            clear_output(wait=True)
            
            # Get actual index after sorting
            actual_index = sorted_indices[dna_index]
            
            dna_info = dnas_for_viz[actual_index]
            sim_result = sims_for_viz[actual_index]
            
            if sim_result is None:
                print(f"❌ No simulation data available for DNA {actual_index + 1}")
                return
            
            # Show if this is a target DNA
            is_target = target_dnas and dna_info in target_dnas
            target_marker = " 🎯 TARGET" if is_target else ""
            
            # Display DNA information
            print(f"🧬 DNA {dna_index + 1} of {len(dnas_for_viz)} (Original Index: {actual_index + 1}){target_marker}")
            print(f"📊 Scores: Original={dna_info['original_score']}, Pruned={dna_info['pruned_score']} "
                  f"(Exp:{dna_info['final_exp_score']}, Cont:{dna_info['final_cont_score']})")
            print(f"⚖️  Weights: {dna_info['original_nonzero']} → {dna_info['pruned_nonzero']} "
                  f"({dna_info['weights_removed']} removed, {dna_info['weights_removed']/dna_info['original_nonzero']*100:.1f}% reduction)")
            
            # Handle different original_dna formats (successful vectors vs pruned results)
            orig_dna = dna_info['original_dna']
            if isinstance(orig_dna, dict):
                # Handle successful vector format
                if 'run_folder' in orig_dna:
                    run_folder = orig_dna['run_folder']
                    generation = orig_dna.get('generation', 'Unknown')
                elif 'original_dna_id' in orig_dna:
                    # This is from a successful vector - get info from high_scoring_dnas if available
                    run_folder = "Successful Vector"
                    generation = orig_dna.get('generation', orig_dna.get('pass', 'Unknown'))
                else:
                    run_folder = "Unknown"
                    generation = "Unknown"
            else:
                run_folder = "Unknown"
                generation = "Unknown"
            
            print(f"📁 Source: {run_folder}, Gen/Pass: {generation}")
            
            if is_target:
                print(f"🎯 TARGET CRITERIA MET:")
                print(f"  Score >= {PRUNING_THRESHOLD}: {dna_info['pruned_score']} ✅")
                print(f"  Weights <= {MAX_PRUNED_WEIGHTS}: {dna_info['pruned_nonzero']} ✅")
            
            print(f"\\n🧬 Pruned DNA Vector:")
            print(f"  {dna_info['pruned_dna']}")
            
            # Show network connections
            G, neu_coords, connections = create_directed_graph_from_dna(dna_info['pruned_dna'], dna_info)
            if connections:
                print(f"\\n🔗 Network Connections ({len(connections)} total):")
                for from_neuron, to_neuron, weight in connections:
                    is_inhibitory = from_neuron in INHIBITORY_NEURONS
                    conn_type = "Inhibitory" if is_inhibitory else "Excitatory"
                    print(f"  {from_neuron:8s} → {to_neuron:8s} | {weight:6d} | {conn_type}")
            else:
                print("\\n⚠️  No connections found in pruned DNA")
            
            # Create and display plots based on selection
            if viz_type in ['voltage', 'both']:
                print("\\n📈 Voltage Traces:")
                voltage_fig = create_voltage_plot(sim_result, dna_info)
                voltage_fig.show()
            
            if viz_type in ['network', 'both']:
                print("\\n🌐 Network Topology:")
                if connections:
                    network_fig = create_network_plot(G, neu_coords, connections, dna_info)
                    plt.show()
                else:
                    print("  ⚠️  Cannot create network graph: No connections to display")
    
    # Set up interactions
    def on_sort_change(change):
        sort_data(change['new'])
        update_plot(dna_slider.value, change['new'], viz_dropdown.value)
    
    def on_slider_change(change):
        update_plot(change['new'], sort_dropdown.value, viz_dropdown.value)
    
    def on_viz_change(change):
        update_plot(dna_slider.value, sort_dropdown.value, change['new'])
    
    sort_dropdown.observe(on_sort_change, names='value')
    dna_slider.observe(on_slider_change, names='value')
    viz_dropdown.observe(on_viz_change, names='value')
    
    # Initial sort
    sort_data(sort_dropdown.value)
    
    # Create layout
    controls = HBox([sort_dropdown, viz_dropdown, dna_slider])
    
    # Display initial plot
    update_plot(0, sort_dropdown.value, viz_dropdown.value)
    
    return VBox([controls, output])

# Create and display the dual DNA browser
if dnas_to_simulate and simulation_results:
    viz_type = "TARGET" if target_dnas else "ALL PRUNED"
    print(f"\\n🎛️ Step 5: Creating interactive dual DNA browser for {viz_type} DNAs...")
    print(f"📊 Browser ready with {len(dnas_to_simulate)} DNA vectors")
    
    if target_dnas:
        print(f"🎯 Showing TARGET DNAs that meet criteria:")
        print(f"  Score >= {PRUNING_THRESHOLD} AND Weights <= {MAX_PRUNED_WEIGHTS}")
    
    print("\\nControls:")
    print("• Sort dropdown: Change ordering of DNAs")
    print("• Show dropdown: Choose between voltage traces, network graph, or both")
    print("• DNA slider: Browse through different DNA solutions")
    print("\\nEach view shows:")
    print("• Voltage traces: Experimental vs control conditions with stimulus markers")
    print("• Network graph: Directed connectivity with edge weights and inhibitory markers")
    print("• Gold borders highlight neurons used for fitness evaluation")
    print("• 🎯 TARGET marker shows DNAs meeting both score and weight criteria\\n")
    
    browser = create_dual_dna_browser()
    display(browser)
else:
    print("❌ Cannot create browser: No data available")
    print("\\nCheck:")
    print("1. Results folder path is correct")
    print("2. Score threshold is appropriate")
    print("3. Aggregated results files exist in subfolders")

\n🎛️ Step 5: Creating interactive dual DNA browser for TARGET DNAs...
📊 Browser ready with 1 DNA vectors
🎯 Showing TARGET DNAs that meet criteria:
  Score >= 975 AND Weights <= 18
\nControls:
• Sort dropdown: Change ordering of DNAs
• Show dropdown: Choose between voltage traces, network graph, or both
• DNA slider: Browse through different DNA solutions
\nEach view shows:
• Voltage traces: Experimental vs control conditions with stimulus markers
• Network graph: Directed connectivity with edge weights and inhibitory markers
• Gold borders highlight neurons used for fitness evaluation
• 🎯 TARGET marker shows DNAs meeting both score and weight criteria\n


## Data Export & Summary

In [10]:
# Save all results for later use, including target DNAs
if pruned_results:
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    export_file = f"multiple_ga_analysis_{timestamp}.pkl"
    
    export_data = {
        'config': {
            'results_folder': RESULTS_FOLDER,
            'score_threshold': SCORE_THRESHOLD,
            'pruning_threshold': PRUNING_THRESHOLD,
            'max_dnas_processed': MAX_DNAS_TO_PROCESS,
            'max_pruned_weights': MAX_PRUNED_WEIGHTS
        },
        'high_scoring_dnas': high_scoring_dnas,
        'pruned_results': pruned_results,
        'target_dnas': target_dnas,  # Add target DNAs to export
        'simulation_results': simulation_results,
        'timestamp': timestamp
    }
    
    with open(export_file, 'wb') as f:
        pickle.dump(export_data, f)
    
    print(f"💾 All results exported to: {export_file}")
    
    # Create summary DataFrame for all pruned results
    summary_data = []
    for i, result in enumerate(pruned_results):
        is_target = target_dnas and result in target_dnas
        
        # Handle different original_dna formats
        orig_dna = result['original_dna']
        if isinstance(orig_dna, dict):
            run_folder = orig_dna.get('run_folder', 'Unknown')
            generation = orig_dna.get('generation', 'Unknown')
            process_id = orig_dna.get('process_id', 'Unknown')
        else:
            run_folder = 'Unknown'
            generation = 'Unknown'
            process_id = 'Unknown'
        
        summary_data.append({
            'DNA_ID': i + 1,
            'Is_Target': '🎯' if is_target else '',
            'Original_Score': result['original_score'],
            'Pruned_Score': result['pruned_score'],
            'Exp_Score': result['final_exp_score'],
            'Cont_Score': result['final_cont_score'],
            'Original_Weights': result['original_nonzero'],
            'Pruned_Weights': result['pruned_nonzero'],
            'Weights_Removed': result['weights_removed'],
            'Reduction_Percent': result['weights_removed'] / result['original_nonzero'] * 100,
            'Run_Folder': run_folder,
            'Generation': generation,
            'Process_ID': process_id
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\n📊 FINAL SUMMARY:")
    print("=" * 60)
    print(f"Results folder analyzed: {RESULTS_FOLDER}")
    print(f"High-scoring DNAs found: {len(high_scoring_dnas)} (threshold: {SCORE_THRESHOLD})")
    print(f"Successfully pruned: {len(pruned_results)}")
    print(f"TARGET DNAs found: {len(target_dnas)} (score >= {PRUNING_THRESHOLD}, weights <= {MAX_PRUNED_WEIGHTS})")
    print(f"Average weight reduction: {summary_df['Reduction_Percent'].mean():.1f}%")
    print(f"Best pruned score: {summary_df['Pruned_Score'].max()}")
    print(f"Most efficient (fewest weights): {summary_df['Pruned_Weights'].min()} weights")
    print(f"Total original weights: {summary_df['Original_Weights'].sum()}")
    print(f"Total pruned weights: {summary_df['Pruned_Weights'].sum()}")
    
    # Save summary CSV
    csv_file = f"multiple_ga_summary_{timestamp}.csv"
    summary_df.to_csv(csv_file, index=False)
    print(f"\n📄 Summary table saved to: {csv_file}")
    
    # Display top results
    print("\n🏆 Top 10 Results by Pruned Score:")
    display(summary_df.nlargest(10, 'Pruned_Score')[['DNA_ID', 'Is_Target', 'Pruned_Score', 'Pruned_Weights', 'Reduction_Percent', 'Run_Folder']])
    
    print("\n🎯 Most Efficient (Fewest Final Weights):")
    display(summary_df.nsmallest(10, 'Pruned_Weights')[['DNA_ID', 'Is_Target', 'Pruned_Score', 'Pruned_Weights', 'Reduction_Percent', 'Run_Folder']])
    
    # Show target DNA summary if any found
    if target_dnas:
        # Create summary for target DNAs with safe field access
        target_summary = []
        for i, target in enumerate(target_dnas):
            orig_dna = target['original_dna']
            if isinstance(orig_dna, dict):
                run_folder = orig_dna.get('run_folder', 'Successful Vector')
                generation = orig_dna.get('generation', orig_dna.get('pass', 'Unknown'))
            else:
                run_folder = 'Unknown'
                generation = 'Unknown'
                
            target_summary.append({
                'Target_ID': i + 1,
                'Pruned_Score': target['pruned_score'],
                'Pruned_Weights': target['pruned_nonzero'],
                'Reduction_Percent': target['weights_removed'] / target['original_nonzero'] * 100 if target['original_nonzero'] > 0 else 0,
                'Run_Folder': run_folder,
                'Generation': generation
            })
        
        target_df = pd.DataFrame(target_summary)
        print(f"\n🎯 TARGET DNAs ({len(target_dnas)} found):")
        display(target_df)
        
        # Save target DNAs separately
        target_file = f"target_dnas_{timestamp}.pkl"
        with open(target_file, 'wb') as f:
            pickle.dump({
                'target_dnas': target_dnas,
                'criteria': {
                    'min_score': PRUNING_THRESHOLD,
                    'max_weights': MAX_PRUNED_WEIGHTS
                },
                'timestamp': timestamp
            }, f)
        print(f"🎯 Target DNAs saved separately to: {target_file}")
    
else:
    print("❌ No results to export")

💾 All results exported to: multiple_ga_analysis_20250904_121819.pkl

📊 FINAL SUMMARY:
Results folder analyzed: results/continuous_runs_H_opt4_thresh970_20250902_173402/successful_run_018
High-scoring DNAs found: 3 (threshold: 970)
Successfully pruned: 3
TARGET DNAs found: 1 (score >= 975, weights <= 18)
Average weight reduction: 72.8%
Best pruned score: 970
Most efficient (fewest weights): 13 weights
Total original weights: 147
Total pruned weights: 40

📄 Summary table saved to: multiple_ga_summary_20250904_121819.csv

🏆 Top 10 Results by Pruned Score:


,DNA_ID,Is_Target,Pruned_Score,Pruned_Weights,Reduction_Percent,Run_Folder
1,2,,970,14,71.428571,successful_run_018
0,1,,969,13,72.916667,successful_run_018
2,3,,969,13,74.000000,successful_run_018



🎯 Most Efficient (Fewest Final Weights):


,DNA_ID,Is_Target,Pruned_Score,Pruned_Weights,Reduction_Percent,Run_Folder
0,1,,969,13,72.916667,successful_run_018
2,3,,969,13,74.000000,successful_run_018
1,2,,970,14,71.428571,successful_run_018



🎯 TARGET DNAs (1 found):


,Target_ID,Pruned_Score,Pruned_Weights,Reduction_Percent,Run_Folder,Generation
0,1,980,18,0.0,Successful Vector,2


🎯 Target DNAs saved separately to: target_dnas_20250904_121819.pkl
